# 🥈 NB_Silver_Analyze_Load_ERP_Tables
Notebook per la promozione delle tabelle ERP dal **Bronze** al **Silver** Lakehouse.

**Flusso:**
1. Lettura file di configurazione CSV
2. Filtro delle sole tabelle ERP
3. Quality check pre-pulizia
4. Pulizia superficiale + scrittura Silver (Full Load)
5. Salvataggio Audit Log

## ⚙ 1. Import & Parametri

In [1]:
# Librerie standard PySpark e Delta Lake
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime
import traceback

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------------------------
# Parametri principali — possono essere sovrascritti da una Pipeline
# BRONZE_LAKEHOUSE : nome del Lakehouse sorgente (layer Bronze)
# SILVER_LAKEHOUSE : nome del Lakehouse destinazione (layer Silver)
# SOURCE_FOLDER    : filtro sul file di config per selezionare solo le tabelle ERP
# CONFIG_TABLE     : nome tabella/file di configurazione degli oggetti da processare
# AUDIT_TABLE      : nome tabella Delta dove salvare il log di ogni esecuzione
# ---------------------------------------------------------------------------
BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"
SOURCE_FOLDER    = "erp"
CONFIG_TABLE     = "pipeline_config"
AUDIT_TABLE      = "silver_audit_log"

# Nomi delle colonne di audit aggiunte a ogni tabella Silver
# Permettono di tracciare quando e da dove è arrivato ogni record
COL_SILVER_TS  = "silver_processed_ts"   # timestamp di scrittura nel Silver
COL_SILVER_SRC = "silver_source_folder"  # cartella sorgente (es. "erp")

# Path relativo del file CSV di configurazione nel Lakehouse corrente (Files/)
config_path = "Files/config_ingestion.csv"

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 3, Finished, Available, Finished, False)

## 📂 2. Recupero Path ABFSS dei Lakehouse

In [2]:
# ---------------------------------------------------------------------------
# In Microsoft Fabric, spark_catalog non supporta la sintassi
# "LH_Bronze.nome_tabella" per letture cross-lakehouse.
# La soluzione è recuperare il path ABFSS (Azure Blob File System)
# di ogni Lakehouse tramite notebookutils, e usarlo nelle read/write Delta.
# ---------------------------------------------------------------------------

bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]
# Esempio risultato:
# abfss://<workspaceId>@onelake.dfs.fabric.microsoft.com/<bronzeId>

silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 4, Finished, Available, Finished, False)

📂 Bronze path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/c9c219f7-14cc-47f2-8994-35d08101e937
📂 Silver path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/b4c95460-8615-46fa-9884-21f085344ad0


## 📋 3. Lettura Configurazione & Selezione Tabelle ERP

In [5]:
# ---------------------------------------------------------------------------
# Il file CSV di configurazione descrive tutti gli oggetti da processare
# nei vari layer. Qui lo leggiamo come DataFrame Spark e filtriamo
# solo le righe dove SourceFolder == "erp", così il notebook è
# riutilizzabile per altri sistemi sorgente cambiando solo SOURCE_FOLDER.
# ---------------------------------------------------------------------------

df_config = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("sep",         ";")
    .csv(config_path)
)

display(df_config)

# Raccogliamo le righe ERP come lista Python (collect) per iterarle nel loop.
# .collect() porta i dati dal cluster Spark al driver: accettabile perché
# la config è piccola (poche decine di righe al massimo).
rows = (
    df_config
    .filter(df_config.SourceFolder == SOURCE_FOLDER)
    .collect()
)

object_names = [row["DestinationTable"] for row in rows]

print(f"📋 Tabelle da processare: {len(object_names)} → {object_names}")

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 288704bc-bfae-41d8-90c7-f3fb4c579579)

📋 Tabelle da processare: 3 → ['bronze_suppliers', 'bronze_inventory', 'bronze_products']


## 🛠 4. Definizione Funzioni

In [6]:
# ---------------------------------------------------------------------------
# FUNZIONE 1: applica_pulizia_superficiale
# Applica una pulizia leggera e non distruttiva su qualsiasi DataFrame.
# Non fa assunzioni sul contenuto: agisce solo su problemi strutturali
# (spazi, stringhe vuote, righe fantasma) senza alterare i valori di business.
# ---------------------------------------------------------------------------
def applica_pulizia_superficiale(df: DataFrame, nome_tabella: str) -> DataFrame:

    # Identifica solo le colonne di tipo stringa per applicare il trim
    # (le colonne numeriche/date non ne hanno bisogno)
    colonne_stringa = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]

    # 1. TRIM: rimuove spazi iniziali e finali da ogni colonna stringa
    #    Es: "  Mario  " → "Mario"
    for col_name in colonne_stringa:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # 2. EMPTY → NULL: normalizza le stringhe vuote come NULL
    #    Es: "" → None — così i tool a valle (Power BI, SQL) le trattano uniformemente
    for col_name in colonne_stringa:
        df = df.withColumn(
            col_name,
            F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))
        )

    # 3. RIGHE FANTASMA: elimina le righe dove ogni singola colonna è NULL
    #    Queste righe non portano informazione e gonfiano i conteggi
    df = df.dropna(how="all")

    # 4. COLONNE DI AUDIT: aggiunge metadati di tracciabilità Silver
    #    Utili per debugging, lineage e monitoraggio delle pipeline
    df = (
        df
        .withColumn(COL_SILVER_TS,  F.current_timestamp())  # quando è stato scritto
        .withColumn(COL_SILVER_SRC, F.lit(SOURCE_FOLDER))   # da quale sistema sorgente
    )

    return df

# ---------------------------------------------------------------------------
# FUNZIONE 2: calcola_quality_report
# Produce un dizionario con le principali metriche di qualità del DataFrame.
# Viene chiamata PRIMA della pulizia per fotografare lo stato raw del Bronze.
# ---------------------------------------------------------------------------
def calcola_quality_report(df: DataFrame, nome_tabella: str) -> dict:

    totale_righe    = df.count()

    # Righe duplicate: differenza tra totale e dopo aver tolto i duplicati
    righe_duplicate = totale_righe - df.dropDuplicates().count()

    # Righe fantasma: righe dove TUTTE le colonne sono NULL
    righe_full_null = totale_righe - df.dropna(how="all").count()

    # Percentuale di NULL per ogni singola colonna
    # Utile per identificare colonne con dati mancanti sistematici
    null_counts = df.select([
        F.round(
            (F.sum(F.col(c).isNull().cast("int")) / totale_righe) * 100, 2
        ).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    return {
        "tabella"         : nome_tabella,
        "totale_righe"    : totale_righe,
        "righe_duplicate" : righe_duplicate,
        "righe_full_null" : righe_full_null,
        "null_pct_per_col": null_counts
    }

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 8, Finished, Available, Finished, False)

## 🔍 5. Quality Check Pre-Pulizia

In [7]:
# ---------------------------------------------------------------------------
# Per ogni tabella ERP, leggiamo il Bronze e stampiamo le metriche di qualità
# PRIMA di qualsiasi trasformazione. Questo ci permette di:
#   - Avere una baseline dello stato raw
#   - Identificare anomalie prima che vadano in Silver
#   - Loggare eventuali problemi per il team dati
# ---------------------------------------------------------------------------
print("=" * 65)
print("🔍 QUALITY CHECK PRE-PULIZIA")
print("=" * 65)

quality_reports = {}

for row in rows:
    nome_file    = row["ObjectName"]        # nome file sorgente originale (es. suppliers.csv)
    nome_tabella = row["DestinationTable"]  # nome tabella Delta nel Bronze (es. bronze_suppliers)

    try:
        # Lettura via path ABFSS per evitare il limite cross-lakehouse di spark_catalog
        df_bronze = (
            spark.read
            .format("delta")
            .load(f"{bronze_path}/Tables/{nome_tabella}")
        )

        report = calcola_quality_report(df_bronze, nome_tabella)
        quality_reports[nome_tabella] = report

        # Stampa strutturata: soglia di attenzione al 10% di NULL per colonna
        print(f"\n📊 Tabella: {nome_tabella.upper()}")
        print(f"   ├── Righe totali    : {report['totale_righe']:,}")
        print(f"   ├── Righe duplicate : {report['righe_duplicate']:,} "
              f"({'⚠ ATTENZIONE' if report['righe_duplicate'] > 0 else '✅ OK'})")
        print(f"   ├── Righe full-null : {report['righe_full_null']:,} "
              f"({'⚠ ATTENZIONE' if report['righe_full_null'] > 0 else '✅ OK'})")
        print(f"   └── Null % per colonna:")
        for col_name, pct in report["null_pct_per_col"].items():
            flag = "⚠" if pct > 10 else "✅"
            print(f"         {flag} {col_name}: {pct}%")

    except Exception as e:
        print(f"\n❌ Errore lettura Bronze.{nome_tabella}: {e}")
        quality_reports[nome_tabella] = {"errore": str(e)}

print("\n" + "=" * 65)

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 9, Finished, Available, Finished, False)

🔍 QUALITY CHECK PRE-PULIZIA

📊 Tabella: BRONZE_SUPPLIERS
   ├── Righe totali    : 500
   ├── Righe duplicate : 0 (✅ OK)
   ├── Righe full-null : 0 (✅ OK)
   └── Null % per colonna:
         ✅ SupplierID: 0.0%
         ✅ SupplierName: 0.0%
         ✅ Country: 0.0%
         ✅ Rating: 0.0%
         ✅ ContractType: 0.0%
         ✅ _ingestion_timestamp: 0.0%
         ✅ _source_file: 0.0%

📊 Tabella: BRONZE_INVENTORY
   ├── Righe totali    : 10,000
   ├── Righe duplicate : 0 (✅ OK)
   ├── Righe full-null : 0 (✅ OK)
   └── Null % per colonna:
         ✅ WarehouseID: 0.0%
         ✅ WarehouseCity: 0.0%
         ✅ ProductID: 0.0%
         ✅ StockQuantity: 0.0%
         ✅ ReservedQuantity: 0.0%
         ✅ AvailableQuantity: 0.0%
         ✅ ReorderLevel: 0.0%
         ✅ RestockQuantity: 0.0%
         ✅ LastRestockDate: 0.0%
         ✅ InventoryStatus: 0.0%
         ✅ _ingestion_timestamp: 0.0%
         ✅ _source_file: 0.0%

📊 Tabella: BRONZE_PRODUCTS
   ├── Righe totali    : 10,000
   ├── Righe d

## 🚀 6. Processing Bronze → Silver (Full Load)

In [8]:
# ---------------------------------------------------------------------------
# Loop principale: per ogni tabella ERP nella configurazione
#   1. Legge la tabella Delta dal Bronze (via path ABFSS)
#   2. Applica la pulizia superficiale
#   3. Sovrascrive la tabella corrispondente nel Silver (Full Load)
#   4. Traccia il risultato in "risultati" per l'audit log
#
# Il prefisso "bronze_" viene sostituito con "silver_" nel nome tabella
# per mantenere la naming convention coerente tra i layer.
# Il try/except garantisce che un errore su una tabella non blocchi le altre.
# ---------------------------------------------------------------------------
risultati = []

print("=" * 65)
print("🚀 INIZIO PROCESSING BRONZE → SILVER")
print("=" * 65)

for row in rows:
    nome_tabella = row["DestinationTable"]
    nome_silver  = nome_tabella.replace("bronze_", "silver_")
    inizio       = datetime.now()

    print(f"\n⏳ Processing: {nome_tabella} → {nome_silver}")

    try:
        # 1. Lettura dal Bronze
        df_bronze    = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella}")
        righe_bronze = df_bronze.count()

        # 2. Pulizia superficiale (trim, empty→null, drop righe fantasma, audit cols)
        df_silver    = applica_pulizia_superficiale(df_bronze, nome_tabella)
        righe_silver = df_silver.count()

        # 3. Scrittura Full nel Silver: overwrite completo ad ogni esecuzione
        #    overwriteSchema:true permette di aggiornare lo schema se cambia nel Bronze
        (
            df_silver.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(f"{silver_path}/Tables/{nome_silver}")
        )

        fine     = datetime.now()
        durata_s = round((fine - inizio).total_seconds(), 2)

        print(f"   ✅ Completato in {durata_s}s")
        print(f"   ├── Righe Bronze  : {righe_bronze:,}")
        print(f"   ├── Righe Silver  : {righe_silver:,}")
        print(f"   └── Scartate      : {righe_bronze - righe_silver:,}")

        risultati.append({
            "tabella"       : nome_silver,
            "righe_bronze"  : righe_bronze,
            "righe_silver"  : righe_silver,
            "righe_scartate": righe_bronze - righe_silver,
            "durata_sec"    : durata_s,
            "stato"         : "SUCCESS",
            "errore"        : "",      # ← stringa vuota: None romperebbe lo schema StringType
            "processed_ts"  : fine
        })

    except Exception as e:
        fine = datetime.now()
        msg  = traceback.format_exc()
        print(f"   ❌ ERRORE su {nome_tabella}: {e}")

        risultati.append({
            "tabella"       : nome_silver,
            "righe_bronze"  : 0,
            "righe_silver"  : 0,
            "righe_scartate": 0,
            "durata_sec"    : round((fine - inizio).total_seconds(), 2),
            "stato"         : "FAILED",
            "errore"        : msg,
            "processed_ts"  : fine
        })
        continue  # prosegui con la tabella successiva senza interrompere il loop

print("\n" + "=" * 65)
print("🏁 PROCESSING COMPLETATO")
print("=" * 65)

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 10, Finished, Available, Finished, False)

🚀 INIZIO PROCESSING BRONZE → SILVER

⏳ Processing: bronze_suppliers → silver_suppliers
   ✅ Completato in 4.71s
   ├── Righe Bronze  : 500
   ├── Righe Silver  : 500
   └── Scartate      : 0

⏳ Processing: bronze_inventory → silver_inventory
   ✅ Completato in 4.17s
   ├── Righe Bronze  : 10,000
   ├── Righe Silver  : 10,000
   └── Scartate      : 0

⏳ Processing: bronze_products → silver_products
   ✅ Completato in 4.35s
   ├── Righe Bronze  : 10,000
   ├── Righe Silver  : 10,000
   └── Scartate      : 0

🏁 PROCESSING COMPLETATO


## 📊 7. Riepilogo & Audit Log

In [9]:
# ---------------------------------------------------------------------------
# Schema esplicito per l'audit log: necessario perché spark.createDataFrame
# non riesce a inferire il tipo di colonne che contengono None/null.
# Definirlo esplicitamente rende il codice robusto in ogni scenario.
# ---------------------------------------------------------------------------
schema_audit = StructType([
    StructField("tabella",        StringType(),    True),
    StructField("righe_bronze",   IntegerType(),   True),
    StructField("righe_silver",   IntegerType(),   True),
    StructField("righe_scartate", IntegerType(),   True),
    StructField("durata_sec",     FloatType(),     True),
    StructField("stato",          StringType(),    True),
    StructField("errore",         StringType(),    True),
    StructField("processed_ts",   TimestampType(), True),
])

# Riepilogo a console
successi = [r for r in risultati if r["stato"] == "SUCCESS"]
falliti  = [r for r in risultati if r["stato"] == "FAILED"]

print(f"\n📊 RIEPILOGO ESECUZIONE")
print(f"   ✅ Tabelle OK     : {len(successi)}/{len(risultati)}")
print(f"   ❌ Tabelle Fallite: {len(falliti)}/{len(risultati)}")

if falliti:
    print(f"\n⚠  TABELLE CON ERRORI:")
    for r in falliti:
        print(f"   → {r['tabella']}: {r['errore'][:120]}...")

# ---------------------------------------------------------------------------
# Salvataggio Audit Log nel Silver in modalità append:
# ogni esecuzione aggiunge nuove righe senza cancellare la storia.
# Questo permette di analizzare trend, durate e anomalie nel tempo.
# ---------------------------------------------------------------------------
df_audit = spark.createDataFrame(risultati, schema=schema_audit)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(f"{silver_path}/Tables/{AUDIT_TABLE}")
)

print(f"\n📝 Audit log salvato in: Silver → {AUDIT_TABLE}")

# Solleva eccezione esplicita se ci sono fallimenti:
# questo permette alla Pipeline Data Factory di rilevare l'errore
# e attivare eventuali notifiche o retry automatici.
if falliti:
    raise Exception(
        f"❌ {len(falliti)} tabelle non processate. "
        f"Consulta '{AUDIT_TABLE}' per i dettagli."
    )

StatementMeta(, 7da0ca55-cad4-4d5e-a3a5-b266c66647ed, 11, Finished, Available, Finished, False)


📊 RIEPILOGO ESECUZIONE
   ✅ Tabelle OK     : 3/3
   ❌ Tabelle Fallite: 0/3

📝 Audit log salvato in: Silver → silver_audit_log
